# 01. Clustering Algorithms: Comparative Analysis and Implementation
**Algorithms Covered:**
1. **K-Means Clustering** (Standard Lloyd's Algorithm)
2. **Modified K-Means** (Bisecting K-Means & K-Means++ Initialization)
3. **Hierarchical Clustering** (Agglomerative Clustering with Dendrogram Visualization)
4. **Fuzzy C-Means (FCM)** (Soft Clustering with Membership Degrees)

**Dataset:** Real-World Mall Customers Dataset (`data/mall_customers.csv`) and Synthetic Benchmark Multi-Cluster Data.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans, BisectingKMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from scipy.cluster.hierarchy import dendrogram, linkage

# Styling setup
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 110

np.random.seed(42)
print("Libraries successfully imported!")


## 1. Data Loading and Exploratory Data Analysis (EDA)
We load the **Mall Customers** dataset, which consists of customer demographics, annual income (in $k), and spending score (1-100).


In [ ]:
df_mall = pd.read_csv('../data/mall_customers.csv')
print("Dataset Shape:", df_mall.shape)
df_mall.head()


In [ ]:
# Summary statistics
print("Missing values per column:\n", df_mall.isnull().sum())
df_mall.describe()


In [ ]:
# Feature Selection & Normalization
# Selecting Annual Income and Spending Score for 2D clustering visualization
feature_cols = ['Annual_Income_k$', 'Spending_Score'] if 'Annual_Income_k$' in df_mall.columns else ['Annual Income (k$)', 'Spending Score (1-100)']
X_mall = df_mall[feature_cols].values

scaler = StandardScaler()
X_mall_scaled = scaler.fit_transform(X_mall)

# Scatter plot of raw data
plt.figure(figsize=(8, 5))
plt.scatter(X_mall[:, 0], X_mall[:, 1], c='royalblue', edgecolors='k', s=50, alpha=0.8)
plt.title("Mall Customers: Annual Income vs Spending Score", fontsize=14, fontweight='bold')
plt.xlabel("Annual Income (k$)", fontsize=12)
plt.ylabel("Spending Score (1-100)", fontsize=12)
plt.tight_layout()
plt.show()


## 2. Algorithm 1: Standard K-Means Clustering
### Mathematical Formulation
Given dataset $X = \{x_1, x_2, \dots, x_n\}$, K-Means partitions the data into $K$ disjoint clusters $C_1, C_2, \dots, C_K$ by minimizing the Within-Cluster Sum of Squares (WCSS / Inertia):
$$J = \sum_{k=1}^K \sum_{x_i \in C_k} \|x_i - \mu_k\|^2$$
where $\mu_k = 
rac{1}{|C_k|}\sum_{x_i \in C_k} x_i$ is the centroid of cluster $C_k$.

### Determining Optimal $K$: Elbow Method & Silhouette Analysis


In [ ]:
k_range = range(2, 11)
wcss = []
silhouette_scores = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    kmeans.fit(X_mall_scaled)
    wcss.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_mall_scaled, kmeans.labels_))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Elbow Curve
ax1.plot(k_range, wcss, 'bo-', linewidth=2, markersize=8)
ax1.set_title("Elbow Method for Optimal K", fontsize=14, fontweight='bold')
ax1.set_xlabel("Number of Clusters (K)", fontsize=12)
ax1.set_ylabel("Inertia (WCSS)", fontsize=12)
ax1.axvline(x=5, color='r', linestyle='--', label='Elbow at K=5')
ax1.legend()

# Silhouette Curve
ax2.plot(k_range, silhouette_scores, 'go-', linewidth=2, markersize=8)
ax2.set_title("Silhouette Scores across K", fontsize=14, fontweight='bold')
ax2.set_xlabel("Number of Clusters (K)", fontsize=12)
ax2.set_ylabel("Silhouette Score", fontsize=12)
ax2.axvline(x=5, color='r', linestyle='--', label='Optimal K=5')
ax2.legend()

plt.tight_layout()
plt.show()


In [ ]:
# Fitting final K-Means model with optimal K=5
optimal_k = 5
kmeans_model = KMeans(n_clusters=optimal_k, init='k-means++', n_init=20, random_state=42)
kmeans_labels = kmeans_model.fit_predict(X_mall_scaled)
kmeans_centers = scaler.inverse_transform(kmeans_model.cluster_centers_)

plt.figure(figsize=(9, 6))
palette = sns.color_palette("deep", optimal_k)
for cluster_id in range(optimal_k):
    cluster_points = X_mall[kmeans_labels == cluster_id]
    plt.scatter(cluster_points[:, 0], cluster_points[:, 1], label=f'Cluster {cluster_id + 1}', s=60, edgecolors='k', alpha=0.85)

plt.scatter(kmeans_centers[:, 0], kmeans_centers[:, 1], c='black', marker='X', s=250, label='Centroids', edgecolors='white', linewidths=2)
plt.title("Standard K-Means Clustering (K=5)", fontsize=14, fontweight='bold')
plt.xlabel("Annual Income (k$)", fontsize=12)
plt.ylabel("Spending Score (1-100)", fontsize=12)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

print(f"K-Means Silhouette Score: {silhouette_score(X_mall_scaled, kmeans_labels):.4f}")
print(f"K-Means Davies-Bouldin Index: {davies_bouldin_score(X_mall_scaled, kmeans_labels):.4f}")


## 3. Algorithm 2: Modified K-Means
Modified variants of K-Means address known limitations of the standard algorithm:
1. **K-Means++ Initialization**: Selects initial cluster centers with probability proportional to the squared Euclidean distance to the nearest existing center, drastically reducing convergence time and sensitivity to local minima.
2. **Bisecting K-Means**: A hierarchical divisive variant that repeatedly splits clusters using 2-means until $K$ clusters are obtained. It produces more uniform and consistent cluster sizes.


In [ ]:
# Comparing K-Means++ vs Random Initialization Stability
n_trials = 30
inertia_random = []
inertia_kpp = []

for seed in range(n_trials):
    km_rnd = KMeans(n_clusters=5, init='random', n_init=1, random_state=seed)
    km_rnd.fit(X_mall_scaled)
    inertia_random.append(km_rnd.inertia_)
    
    km_kpp = KMeans(n_clusters=5, init='k-means++', n_init=1, random_state=seed)
    km_kpp.fit(X_mall_scaled)
    inertia_kpp.append(km_kpp.inertia_)

plt.figure(figsize=(9, 5))
plt.boxplot([inertia_random, inertia_kpp], tick_labels=['Random Init', 'K-Means++ Init'], patch_artist=True)
plt.title("Inertia Variance: Random vs K-Means++ Initialization (30 Runs)", fontsize=14, fontweight='bold')
plt.ylabel("Inertia (Lower & More Consistent is Better)", fontsize=12)
plt.tight_layout()
plt.show()

print(f"Random Init Mean Inertia: {np.mean(inertia_random):.2f} (std: {np.std(inertia_random):.2f})")
print(f"K-Means++ Mean Inertia:  {np.mean(inertia_kpp):.2f} (std: {np.std(inertia_kpp):.2f})")


In [ ]:
# Bisecting K-Means Execution
bisecting_km = BisectingKMeans(n_clusters=optimal_k, random_state=42, bisecting_strategy='biggest_inertia')
bisecting_labels = bisecting_km.fit_predict(X_mall_scaled)
bisecting_centers = scaler.inverse_transform(bisecting_km.cluster_centers_)

plt.figure(figsize=(9, 6))
for cluster_id in range(optimal_k):
    cluster_points = X_mall[bisecting_labels == cluster_id]
    plt.scatter(cluster_points[:, 0], cluster_points[:, 1], label=f'Bisecting Cluster {cluster_id + 1}', s=60, edgecolors='k', alpha=0.85)

plt.scatter(bisecting_centers[:, 0], bisecting_centers[:, 1], c='darkred', marker='D', s=200, label='Centroids', edgecolors='white', linewidths=2)
plt.title("Bisecting K-Means Clustering (Divisive Strategy)", fontsize=14, fontweight='bold')
plt.xlabel("Annual Income (k$)", fontsize=12)
plt.ylabel("Spending Score (1-100)", fontsize=12)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

print(f"Bisecting K-Means Silhouette Score: {silhouette_score(X_mall_scaled, bisecting_labels):.4f}")
print(f"Bisecting K-Means Davies-Bouldin Index: {davies_bouldin_score(X_mall_scaled, bisecting_labels):.4f}")


## 4. Algorithm 3: Hierarchical Clustering (Agglomerative)
### Mathematical Formulation
Agglomerative clustering begins with each point as its own cluster and iteratively merges the closest pair of clusters until only one cluster remains.
Using **Ward's Linkage**, the distance between cluster $A$ and cluster $B$ measures the increase in total within-cluster variance upon merging:
$$\Delta W(A, B) = 
rac{n_A n_B}{n_A + n_B} \|\mu_A - \mu_B\|^2$$


In [ ]:
# Computing Linkage Matrix and Generating Dendrogram
linkage_matrix = linkage(X_mall_scaled, method='ward')

plt.figure(figsize=(14, 7))
dendrogram(
    linkage_matrix,
    truncate_mode='lastp',
    p=25,
    leaf_rotation=45,
    leaf_font_size=10,
    show_contracted=True
)
plt.title("Hierarchical Clustering Dendrogram (Ward Linkage)", fontsize=14, fontweight='bold')
plt.xlabel("Cluster Sample Index / Merged Size", fontsize=12)
plt.ylabel("Euclidean Distance (Ward Threshold)", fontsize=12)
plt.axhline(y=10.0, color='r', linestyle='--', label='Cut-off Threshold (K=5)')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Fitting Agglomerative Clustering with K=5
agg_model = AgglomerativeClustering(n_clusters=optimal_k, metric='euclidean', linkage='ward')
agg_labels = agg_model.fit_predict(X_mall_scaled)

plt.figure(figsize=(9, 6))
for cluster_id in range(optimal_k):
    cluster_points = X_mall[agg_labels == cluster_id]
    plt.scatter(cluster_points[:, 0], cluster_points[:, 1], label=f'Cluster {cluster_id + 1}', s=60, edgecolors='k', alpha=0.85)

plt.title("Hierarchical Agglomerative Clustering (K=5, Ward Linkage)", fontsize=14, fontweight='bold')
plt.xlabel("Annual Income (k$)", fontsize=12)
plt.ylabel("Spending Score (1-100)", fontsize=12)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

print(f"Hierarchical Silhouette Score: {silhouette_score(X_mall_scaled, agg_labels):.4f}")
print(f"Hierarchical Davies-Bouldin Index: {davies_bouldin_score(X_mall_scaled, agg_labels):.4f}")


## 5. Algorithm 4: Fuzzy C-Means (FCM)
### Mathematical Formulation
Unlike hard clustering (K-Means), Fuzzy C-Means assigns each sample $x_i$ a **fuzzy membership degree** $u_{ik} \in [0, 1]$ across all $K$ clusters such that $\sum_{k=1}^K u_{ik} = 1$.
The objective function is:
$$J_m = \sum_{i=1}^N \sum_{k=1}^K u_{ik}^m \|x_i - v_k\|^2, \quad m > 1$$
Where:
- $m$ is the fuzzifier (typically $m=2.0$).
- Centroid update: $v_k = 
rac{\sum_{i=1}^N u_{ik}^m x_i}{\sum_{i=1}^N u_{ik}^m}$
- Membership update: $u_{ik} = 
rac{1}{\sum_{j=1}^K \left(
rac{\|x_i - v_k\|}{\|x_i - v_j\|}
ight)^{
rac{2}{m-1}}}$


In [ ]:
class FuzzyCMeans:
    def __init__(self, n_clusters=5, m=2.0, max_iter=150, tol=1e-5, random_state=42):
        self.n_clusters = n_clusters
        self.m = m
        self.max_iter = max_iter
        self.tol = tol
        self.random_state = random_state

    def fit(self, X):
        rng = np.random.RandomState(self.random_state)
        n_samples = X.shape[0]
        # Random initial membership matrix U normalized per row
        U = rng.rand(n_samples, self.n_clusters)
        U = U / U.sum(axis=1, keepdims=True)

        for iteration in range(self.max_iter):
            U_old = U.copy()
            # Update cluster centers
            Um = U ** self.m
            centers = np.dot(Um.T, X) / Um.sum(axis=0)[:, np.newaxis]
            
            # Compute distance matrix (n_samples, n_clusters)
            dist = np.linalg.norm(X[:, np.newaxis, :] - centers[np.newaxis, :, :], axis=2)
            dist = np.fmax(dist, 1e-10) # avoid division by zero
            
            # Update membership matrix
            inv_dist = 1.0 / dist
            power = 2.0 / (self.m - 1.0)
            inv_dist_p = inv_dist ** power
            U = inv_dist_p / inv_dist_p.sum(axis=1, keepdims=True)

            if np.max(np.abs(U - U_old)) < self.tol:
                break

        self.cluster_centers_ = centers
        self.u_ = U
        self.labels_ = np.argmax(U, axis=1)
        return self

fcm = FuzzyCMeans(n_clusters=optimal_k, m=2.0, random_state=42)
fcm.fit(X_mall_scaled)
fcm_labels = fcm.labels_
fcm_centers = scaler.inverse_transform(fcm.cluster_centers_)

plt.figure(figsize=(9, 6))
for cluster_id in range(optimal_k):
    cluster_points = X_mall[fcm_labels == cluster_id]
    plt.scatter(cluster_points[:, 0], cluster_points[:, 1], label=f'FCM Cluster {cluster_id + 1}', s=60, edgecolors='k', alpha=0.85)

plt.scatter(fcm_centers[:, 0], fcm_centers[:, 1], c='gold', marker='*', s=350, label='FCM Centroids', edgecolors='k', linewidths=2)
plt.title("Fuzzy C-Means (FCM) Hard Partition Representation", fontsize=14, fontweight='bold')
plt.xlabel("Annual Income (k$)", fontsize=12)
plt.ylabel("Spending Score (1-100)", fontsize=12)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

print(f"FCM Silhouette Score: {silhouette_score(X_mall_scaled, fcm_labels):.4f}")
print(f"FCM Davies-Bouldin Index: {davies_bouldin_score(X_mall_scaled, fcm_labels):.4f}")


In [ ]:
# Visualizing Soft Membership Degrees
# Plotting maximum membership probability distribution
max_memberships = np.max(fcm.u_, axis=1)

plt.figure(figsize=(10, 6))
scatter = plt.scatter(X_mall[:, 0], X_mall[:, 1], c=max_memberships, cmap='plasma', s=70, edgecolors='gray', alpha=0.9)
cbar = plt.colorbar(scatter)
cbar.set_label('Max Membership Certainty degree (max $u_{ik}$)', fontsize=12)
plt.title("FCM Soft Membership Certainty Map", fontsize=14, fontweight='bold')
plt.xlabel("Annual Income (k$)", fontsize=12)
plt.ylabel("Spending Score (1-100)", fontsize=12)
plt.tight_layout()
plt.show()


## 6. Comprehensive Quantitative Comparison
We evaluate all four clustering algorithms across three primary unsupervised metrics:
1. **Silhouette Score** (Higher is better: ranges from -1 to 1; measures intra-cluster cohesion vs. inter-cluster separation)
2. **Davies-Bouldin Index** (Lower is better: measures average similarity between clusters)
3. **Calinski-Harabasz Score** (Higher is better: ratio of between-clusters dispersion to within-cluster dispersion)


In [ ]:
models = {
    "Standard K-Means": kmeans_labels,
    "Bisecting K-Means": bisecting_labels,
    "Hierarchical (Ward)": agg_labels,
    "Fuzzy C-Means": fcm_labels
}

results = []
for name, labels in models.items():
    results.append({
        "Algorithm": name,
        "Silhouette Score": silhouette_score(X_mall_scaled, labels),
        "Davies-Bouldin Index": davies_bouldin_score(X_mall_scaled, labels),
        "Calinski-Harabasz Score": calinski_harabasz_score(X_mall_scaled, labels)
    })

df_comparison = pd.DataFrame(results).set_index("Algorithm")
print("Clustering Performance Comparison on Mall Customers:")
display(df_comparison.style.highlight_max(subset=['Silhouette Score', 'Calinski-Harabasz Score'], color='lightgreen')
                           .highlight_min(subset=['Davies-Bouldin Index'], color='lightgreen'))


In [ ]:
# Bar chart comparison of metrics
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

df_comparison['Silhouette Score'].plot(kind='bar', ax=axes[0], color='teal', edgecolor='k')
axes[0].set_title("Silhouette Score (Higher is Better)", fontweight='bold')
axes[0].set_ylabel("Score")
axes[0].tick_params(axis='x', rotation=30)

df_comparison['Davies-Bouldin Index'].plot(kind='bar', ax=axes[1], color='coral', edgecolor='k')
axes[1].set_title("Davies-Bouldin Index (Lower is Better)", fontweight='bold')
axes[1].set_ylabel("Score")
axes[1].tick_params(axis='x', rotation=30)

df_comparison['Calinski-Harabasz Score'].plot(kind='bar', ax=axes[2], color='purple', edgecolor='k')
axes[2].set_title("Calinski-Harabasz Score (Higher is Better)", fontweight='bold')
axes[2].set_ylabel("Score")
axes[2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()


## 7. Conclusions & Algorithm Selection Insights
- **Standard K-Means** provides an excellent balance of speed and cluster separation on globular, well-separated data.
- **Bisecting K-Means** prevents small outlier clusters from dominating the solution and yields stable, balanced partitions.
- **Hierarchical Clustering** requires no prior assumption about $K$ when constructing the tree, and the dendrogram offers visual interpretability into cluster hierarchy.
- **Fuzzy C-Means** captures overlapping clusters and data ambiguity, providing actionable membership probabilities for points along boundary regions.
